# 11 - Notebook suy luận độc lập (S5-06)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (`COVID`, `Lung_Opacity`, `Normal`, `Viral Pneumonia`).  
**Người phụ trách:** Thái Tú (duyệt Vũ Thương).

## Mô hình đóng băng
`StandardScaler` (chỉ fit trên train) + `SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced', probability=True, random_state=42)` trên **152 đặc trưng thủ công** (Histogram 64 + GLCM 60 + LBP 28) của ảnh gốc xám 299×299 (`orig_to_orig`, cấu hình trong `results/final_test/config.yaml`).

## Notebook này làm gì
1. Dựng môi trường (cục bộ hoặc phiên Kaggle mới) và tìm mã nguồn `src/`.
2. Kiểm tra SHA-256 của `split_v1.csv`, rồi **fit lại mô hình chỉ trên tập train** (hoặc nạp bộ nhớ đệm cục bộ nếu hợp lệ).
3. Nạp **một ảnh X-quang bất kỳ**, trích đúng 152 đặc trưng bằng hàm chuẩn trong `src/`.
4. Dự đoán, hiển thị **xác suất 4 lớp**, nhãn dự đoán và biểu đồ phân phối xác suất.

## Ràng buộc kiểm soát
- **Cổng H7:** chỉ dùng học máy truyền thống, không dùng framework deep learning.
- **Cổng H2:** notebook chỉ đọc tập `train` (để fit) và tùy chọn tập `val` (kiểm tra tái lập); không bao giờ đọc tập kiểm tra cuối cùng.
- **Quy tắc D4:** chỉ báo cáo số đo thực tế. Mọi lời giải thích về sự khác biệt giữa val và test đều là giả thuyết chưa kiểm chứng, không nêu nguyên nhân.

## 1. Thiết lập môi trường và tìm mã nguồn

- **Cục bộ:** chạy từ thư mục gốc repo (hoặc `notebooks/`); đặt biến môi trường `CXR_DATA_ROOT` trỏ tới thư mục chứa `COVID/images`.
- **Kaggle (phiên mới tinh):** thêm hai Dataset vào notebook: bộ ảnh `covid19-radiography-database` và gói mã riêng của notebook này. Tạo gói mã từ repo (chạy ở thư mục gốc repo):

  `python -c "from src.inference_demo import build_inference_code_package as b; b('.', 'outputs/inference_pkg', '<commit>')"`  
  `kaggle datasets create -p outputs/inference_pkg`

  Gói gồm `COMMIT`, `src/*.py` (có `inference_demo.py`) và `split_v1.csv`. **Không** gắn gói đánh giá test đã niêm phong (`cxr-final-test-code` / `cxr-final-test-dry-code`): các gói đó không có `inference_demo.py` nên notebook sẽ từ chối.

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

INPUT_ROOT = Path(os.environ.get('KAGGLE_INPUT_PATH', '/kaggle/input'))
WORKING_ROOT = Path(os.environ.get('KAGGLE_WORKING_PATH', '/kaggle/working'))


def _find_repo_root():
    for cand in (Path.cwd(), Path.cwd().parent):
        if (cand / 'src' / 'inference_demo.py').is_file():
            return cand
    return None


def _find_kaggle_code_package(input_root: Path) -> Path:
    patterns = ['COMMIT', '*/COMMIT', '*/*/COMMIT', '*/*/*/COMMIT', '*/*/*/*/COMMIT']
    for pat in patterns:
        for marker in sorted(input_root.glob(pat)):
            pkg = marker.parent
            if (pkg / 'src' / 'inference_demo.py').is_file() or (pkg / 'inference_demo.py').is_file():
                return pkg
    raise FileNotFoundError(
        f'Khong tim thay goi ma nguon (co COMMIT va inference_demo.py) trong {input_root}. '
        'Hay them Dataset goi ma vao notebook.'
    )


REPO_ROOT = _find_repo_root()
if REPO_ROOT is not None:
    CODE_ROOT = REPO_ROOT
    SPLIT_CSV = REPO_ROOT / 'splits' / 'split_v1.csv'
    ENV_NAME = 'local'
else:
    pkg_dir = _find_kaggle_code_package(INPUT_ROOT)
    CODE_ROOT = WORKING_ROOT / 'code'
    (CODE_ROOT / 'src').mkdir(parents=True, exist_ok=True)
    src_dir = pkg_dir / 'src' if (pkg_dir / 'src').is_dir() else pkg_dir
    for py_file in src_dir.glob('*.py'):
        shutil.copyfile(py_file, CODE_ROOT / 'src' / py_file.name)
    SPLIT_CSV = pkg_dir / 'split_v1.csv'
    ENV_NAME = 'kaggle'

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.inference_demo import (
    extract_152_features,
    ensure_not_heldout_image,
    fit_or_load_model,
    load_image_grayscale_299,
    plot_probabilities,
    predict_image,
    verify_on_val,
)
from src.rq2_runner import find_data_root

print(f'Moi truong: {ENV_NAME} | ma nguon: {CODE_ROOT}')
assert SPLIT_CSV.is_file(), f'Thieu {SPLIT_CSV}'

## 2. Tham số

- `IMAGE_PATH`: đường dẫn một ảnh X-quang xám 299×299 (PNG). Để `None` thì dùng ảnh mẫu đầu tiên của tập `val` (nhãn tham chiếu chỉ để đối chiếu).
- `RESIZE_TO_299`: nếu `True` (mặc định), ảnh có kích thước khác 299×299 được đổi cỡ (`INTER_AREA`) kèm cảnh báo; mô hình chỉ được kiểm chứng trên ảnh 299×299 của bộ dữ liệu này, nên kết quả trên ảnh nguồn khác không có giá trị đánh giá. Đặt `False` để bắt buộc đúng 299×299.
- `IMAGE_PATH` không được trùng một ảnh thuộc tập giữ lại (test): notebook sẽ từ chối (cổng H2).
- `RUN_VAL_CHECK`: đặt `True` để kiểm tra tái lập Macro-F1 trên `val` (đọc 3174 ảnh, tốn thêm thời gian; cổng G5: |Δ| ≤ 0.002 so với 0.9024).

In [ ]:
IMAGE_PATH = None
RESIZE_TO_299 = True
RUN_VAL_CHECK = False
CACHE_PATH = WORKING_ROOT / 'cxr_inference_model_cache.joblib' if ENV_NAME == 'kaggle' else CODE_ROOT / 'outputs' / 'inference_model_cache.joblib'

try:
    if os.environ.get('CXR_DATA_ROOT'):
        DATA_ROOT = Path(os.environ['CXR_DATA_ROOT'])
    else:
        DATA_ROOT = find_data_root(INPUT_ROOT)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        'Khong tim thay thu muc anh COVID-19 Radiography. Tren may cuc bo hay dat bien moi truong '
        'CXR_DATA_ROOT tro toi thu muc chua COVID/images; tren Kaggle hay them Dataset '
        'covid19-radiography-database.'
    ) from exc
assert (DATA_ROOT / 'COVID' / 'images').is_dir(), f'DATA_ROOT sai: {DATA_ROOT}'
print(f'Thu muc anh: {DATA_ROOT}')

## 3. Dựng mô hình đóng băng (chỉ fit trên train)

`fit_or_load_model` kiểm tra SHA-256 và kích thước các split (G1), rồi nạp bộ nhớ đệm cục bộ nếu khớp SHA-256 split, phiên bản scikit-learn và siêu tham số; nếu không thì fit lại trên 14815 ảnh train (thời gian chạy chưa được đo trong notebook này). Bộ nhớ đệm chỉ do chính notebook này ghi, không đưa vào git.

In [ ]:
scaler, clf, info = fit_or_load_model(SPLIT_CSV, DATA_ROOT, cache_path=CACHE_PATH)
print(f"Nguon mo hinh      : {info['source']}")
print(f"So anh train       : {info['n_train']}")
print(f"SHA-256 split      : {info['split_sha256']}")
print(f"Sieu tham so SVC   : {info['svm_params']}")
print(f"Thu tu lop (classes_): {list(clf.classes_)}")

## 4. Kiểm tra tái lập trên val (tùy chọn)

Chỉ chạy khi `RUN_VAL_CHECK = True`. Bước này chỉ đọc tập `val`.

In [ ]:
if RUN_VAL_CHECK:
    val_f1 = verify_on_val(scaler, clf, SPLIT_CSV, DATA_ROOT)
    print(f'Macro-F1 val tai lap = {val_f1:.4f} (tham chieu 0.9024, dung sai 0.002): DAT G5')
else:
    print('Bo qua kiem tra val (RUN_VAL_CHECK = False).')

## 5. Nạp ảnh X-quang đầu vào

In [ ]:
split_df = pd.read_csv(SPLIT_CSV)
reference_label = None
if IMAGE_PATH is None:
    sample_row = split_df[split_df['split'] == 'val'].iloc[0]
    image_path = DATA_ROOT / sample_row['rel_image_path']
    reference_label = str(sample_row['label'])
else:
    image_path = Path(IMAGE_PATH)
    ensure_not_heldout_image(image_path, SPLIT_CSV)

img = load_image_grayscale_299(image_path, resize=RESIZE_TO_299)
print(f'Anh dau vao: {image_path}')
print(f'Kich thuoc: {img.shape}, kieu: {img.dtype}')
if reference_label is not None:
    print(f'Nhan tham chieu (anh mau tu val): {reference_label}')

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('Anh X-quang dau vao')
plt.axis('off')
plt.show()

## 6. Trích 152 đặc trưng thủ công

Histogram 64 + GLCM 60 + LBP 28 = 152 chiều, bằng hàm chuẩn `extract_ghep` trong `src/features_hog_ghep.py`.

In [ ]:
feats = extract_152_features(img)
assert feats.shape == (152,)
print(f'Vector dac trung: shape = {feats.shape}')
print(f'  Histogram : {feats[:64].shape[0]} chieu (tong = {feats[:64].sum():.4f})')
print(f'  GLCM      : {feats[64:124].shape[0]} chieu')
print(f'  LBP       : {feats[124:].shape[0]} chieu')
print(f'  Gia tri huu han: {bool(np.isfinite(feats).all())}')

## 7. Dự đoán và phân phối xác suất 4 lớp

Nhãn dự đoán lấy từ `clf.predict()` (đúng như khi tính các số đo đã công bố). Do hiệu chỉnh xác suất kiểu Platt của `SVC(probability=True)`, nhãn này đôi khi khác lớp có xác suất cao nhất (argmax) trong biểu đồ.

In [ ]:
result = predict_image(scaler, clf, feats)
proba_table = pd.DataFrame(
    {'Lop': list(result['probabilities'].keys()), 'Xac suat': list(result['probabilities'].values())}
)
print(proba_table.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f"\nNhan du doan: {result['label']} (xac suat {result['probabilities'][result['label']]:.4f})")
print(f"Tong xac suat 4 lop = {sum(result['probabilities'].values()):.6f}")

In [ ]:
fig = plot_probabilities(result)
plt.show()

## 8. Số đo thực tế của mô hình và lưu ý

Số đo đã công bố của mô hình đóng băng ở S5-04 (tập test 3176 ảnh, đánh giá đúng một lần trên Kaggle): Macro-F1 = **0.8714** (khoảng tin cậy 95% bootstrap: 0.8579–0.8840); Macro-F1 trên val = **0.9024**; chênh lệch Test − Val = **-0.0310**. Notebook này không đo lại các con số đó: chỉ khi đặt `RUN_VAL_CHECK = True` nó mới tái lập Macro-F1 trên val (cổng G5).

Mọi lời giải thích về sự khác biệt giữa val và test đều là giả thuyết chưa kiểm chứng, không nêu nguyên nhân.

> **Cảnh báo y khoa:** Đây là sản phẩm học thuật minh họa kỹ thuật học máy truyền thống, **không phải công cụ chẩn đoán**. Kết quả dự đoán không được dùng để ra quyết định lâm sàng. Nhầm lẫn giữa `Normal` và `Lung_Opacity` là dạng lỗi phổ biến nhất trên tập test (304 trên tổng 427 lỗi).